# CAS Exam 5: Frequency-Severity Techniques and Disposal Rate Method

**Source:** Friedland, J. *Estimating Unpaid Claims Using Basic Techniques*, Casualty Actuarial Society, 2010

This notebook demonstrates:
1. Frequency-Severity Technique #1
2. Frequency-Severity Technique #2
3. Disposal Rate Method

Data used from `chainladder/utils/data`:
- `friedland_xyz_freq_sev.csv`
- `friedland_xyz_disp.csv`
- `xyz.csv` (premium used as an exposure proxy for Technique #2)

## Formula Sheet Reference

### Frequency-Severity Technique #1: Develop Counts and Severity Separately

$$\text{Ultimate Claims}_w = \text{Ultimate Counts}_w \times \text{Ultimate Severity}_w$$

**Where:**
- Ultimate Counts_w = Latest Closed Counts_w × CDF_counts(d → ultimate)
- Ultimate Severity_w = Latest Reported Severity_w × CDF_severity(d → ultimate)
- IBNR = Ultimate Claims_w − Latest Reported Claims_w

**Data requirements:** Cumulative closed claim count triangle + reported severity triangle + latest reported claims for IBNR calculation.

### Frequency-Severity Technique #2: Trend-Projected Frequency and Severity

$$\text{Frequency}_w = \frac{\text{Ultimate Counts}_w}{\text{Exposure}_w}$$

For mature AYs (development age ≥ 84 months), fit a log-linear trend:

$$\ln(\text{Frequency}_w) = \hat{a} + \hat{b} \times w \implies \text{Projected Frequency}_w = e^{\hat{a} + \hat{b} w}$$

$$\ln(\text{Severity}_w) = \hat{c} + \hat{d} \times w \implies \text{Projected Severity}_w = e^{\hat{c} + \hat{d} w}$$

$$\text{Ultimate Claims}_w = \text{Exposure}_w \times \text{Projected Frequency}_w \times \text{Projected Severity}_w$$

**Advantage over Technique #1:** Immature AY projections are driven by trend extrapolation, not by potentially unreliable development from sparse early data.

### Disposal Rate Method

$$\text{DR}(w, d) = \frac{\text{Cumulative Closed Counts}_{w,d}}{\text{Ultimate Closed Counts}_w}$$

Rearranged to solve for ultimate:

$$\text{Ultimate Closed Counts}_w = \frac{\text{Cumulative Closed}_{w,d}}{\text{DR}(d)}$$

**Projection:**

$$\text{Projected Incremental Closed}_{d \to d+1} = \text{Ultimate Closed}_w \times \left(\text{DR}_{d+1} - \text{DR}_d\right)$$

$$\text{Projected Incremental Paid}_{d \to d+1} = \text{Projected Inc. Closed} \times \text{Incremental Severity}_{d \to d+1}$$

$$\text{Ultimate Paid}_w = \text{Latest Paid}_w + \sum_{\text{future ages}} \text{Projected Incremental Paid}$$

### When to Use Each Method

| Method | Best Conditions | Avoid When |
|---|---|---|
| Freq-Sev Technique #1 | Count and severity triangles are credible and stable; closed-count development is mature | Severity is volatile at late ages; count triangle is sparse |
| Freq-Sev Technique #2 | Immature AYs make Technique #1 unreliable; exposure base is available; trend data is credible | Frequency/severity trends are unstable; fewer than 4–5 mature AYs for calibration |
| Disposal Rate | Closed-count emergence is credible; incremental severity is stable by development age | Long-tailed lines with very few counts at late ages; incremental severity is erratic |

### Key Assumptions

1. Count triangle follows a stable cumulative development pattern (all three methods)
2. Severity triangle reflects stable cost levels (Freq-Sev Technique #1)
3. Disposal rates are consistent across origin years at the same development age (Disposal Rate)
4. Incremental severity by development age is stable across origin years (Disposal Rate)
5. Frequency and severity trends are log-linear and stable in the calibration window (Technique #2)

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import chainladder as cl

ROOT = Path.cwd().resolve()


DATA_DIR = ROOT / 'chainladder-python' / 'chainladder' / 'utils' / 'data'
freq_sev_df = pd.read_csv(DATA_DIR / 'friedland_xyz_freq_sev.csv')
disp_df = pd.read_csv(DATA_DIR / 'friedland_xyz_disp.csv')
xyz_df = pd.read_csv(DATA_DIR / 'xyz.csv')

freq_sev_triangle = cl.Triangle(
    freq_sev_df,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Closed Claim Counts', 'Reported Claim Counts', 'Reported Claims', 'Reported Severities'],
    cumulative=True,
)

closed_count_triangle = freq_sev_triangle['Closed Claim Counts']
reported_claims_triangle = freq_sev_triangle['Reported Claims']
reported_severity_triangle = freq_sev_triangle['Reported Severities']

{'freq_sev_triangle_shape': freq_sev_triangle.shape, 'valuation_date': str(freq_sev_triangle.valuation_date)}


In [ ]:
latest_counts = closed_count_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_claims = reported_claims_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_severity = reported_severity_triangle.latest_diagonal.to_frame().iloc[:, 0]

closed_long = closed_count_triangle.to_frame(origin_as_datetime=False, keepdims=True).reset_index()
closed_matrix = closed_long.pivot(index='origin', columns='development', values='Closed Claim Counts').sort_index().sort_index(axis=1)
latest_age = closed_matrix.notna().iloc[:, ::-1].idxmax(axis=1)
latest_age.index = pd.to_datetime(latest_age.index.astype(str) + "-01-01")
latest_age = latest_age.reindex(latest_counts.index)

triangle_snapshot = pd.DataFrame(
    {
        'LatestClosedCounts': latest_counts.values,
        'LatestReportedClaims': latest_claims.values,
        'LatestReportedSeverity': latest_severity.values,
        'LatestMaturityAge': latest_age.values,
    },
    index=latest_counts.index.year,
)
triangle_snapshot.index.name = 'AccidentYear'
triangle_snapshot


## Pre-Analysis: Count Triangle Quality Check

Before applying any frequency-severity or disposal rate method, verify that the closed count triangle has stable development. The `link_ratio_table` function produces the standard link ratio exhibit for the count triangle, and percent-closed-by-age is computed as `1 - 1 / vol_wtd_ldf`.

In [ ]:
from reservingengine.reserving import link_ratio_table

lr = link_ratio_table(closed_count_triangle)
pct_closed = 1.0 - 1.0 / lr.loc["Vol Wtd Avg"].astype(float)

print("Count triangle link ratio exhibit:")
display(lr.style.format("{:.4f}", na_rep="---"))

print("\nPercent closed by development age (1 - 1/VolWtd LDF):")
pct_closed.to_frame("percent_closed").style.format("{:.1%}")

**EXAM RED FLAG —** If `percent_closed` at the latest development age is below 90%, there is significant remaining count development. The tail of the disposal rate method depends entirely on how you handle those late-age claims — a poor tail severity assumption can be a larger source of reserve error than the bulk of the projection.

**EXAM RED FLAG —** If `percent_closed` reaches ≥ 98% by the latest development age, the count tail is nearly complete and both the disposal rate method and Frequency-Severity Technique #1 are well-supported by the data.

**Freq-Sev convergence check:** If reported severity link ratios are near 1.000 from early development ages (severity appears fully developed quickly), but paid claims continue to develop (more claims are closing at later ages), then counts are driving ALL remaining development. In this case, Frequency-Severity Technique #1 and the Disposal Rate method will produce similar results — a useful sanity check when both are available.

See `exam5_diagnostics.ipynb` Section 9 for full count triangle diagnostic walkthrough.

## Frequency-Severity Technique #1

Approach:
1. Apply development method to cumulative closed claim counts.
2. Apply development method to reported severities.
3. Multiply projected ultimate counts x projected ultimate severity.
4. Compute IBNR as projected ultimate claims minus latest reported claims.

Assumption emphasis:
- Count and severity patterns observed to date remain representative for future development.

In [ ]:
count_dev = cl.Development(average='volume', n_periods=-1).fit_transform(closed_count_triangle)
severity_dev = cl.Development(average='volume', n_periods=-1).fit_transform(reported_severity_triangle)

count_model = cl.Chainladder().fit(count_dev)
severity_model = cl.Chainladder().fit(severity_dev)

ultimate_counts_t1 = count_model.ultimate_.to_frame().iloc[:, 0]
ultimate_severity_t1 = severity_model.ultimate_.to_frame().iloc[:, 0]
ultimate_claims_t1 = ultimate_counts_t1 * ultimate_severity_t1
ibnr_t1 = ultimate_claims_t1 - latest_claims

tech1_ay = pd.DataFrame(
    {
        'UltimateCounts_T1': ultimate_counts_t1.values,
        'UltimateSeverity_T1': ultimate_severity_t1.values,
        'UltimateClaims_T1': ultimate_claims_t1.values,
        'LatestReportedClaims': latest_claims.values,
        'IBNR_T1': ibnr_t1.values,
    },
    index=ultimate_counts_t1.index.year,
)
tech1_ay.index.name = 'AccidentYear'
tech1_ay.loc['Total'] = tech1_ay.sum()
tech1_ay


## Frequency-Severity Technique #2

Approach:
1. Start from projected ultimate counts.
2. Convert to frequency by dividing by exposure.
3. Select frequency and severity using trend analysis (especially for immature years).
4. Project ultimate claims as: Exposure x Selected Frequency x Selected Severity.

In this demo, premium from `xyz.csv` is used as an exposure proxy.


In [ ]:
exposure_proxy = xyz_df.groupby('AccidentYear')['Premium'].max().sort_index()
exposure_proxy.index = pd.to_datetime(exposure_proxy.index.astype(str) + "-01-01")
exposure_proxy = exposure_proxy.reindex(ultimate_counts_t1.index)

frequency_from_t1 = ultimate_counts_t1 / exposure_proxy
severity_from_t1 = ultimate_severity_t1.copy()

mature_mask = latest_age >= 84
ay_numeric = pd.Series(ultimate_counts_t1.index.year, index=ultimate_counts_t1.index, dtype=float)

freq_fit_mask = mature_mask & frequency_from_t1.notna() & (frequency_from_t1 > 0)
sev_fit_mask = mature_mask & severity_from_t1.notna() & (severity_from_t1 > 0)

if int(freq_fit_mask.sum()) >= 2:
    freq_slope, freq_intercept = np.polyfit(
        ay_numeric[freq_fit_mask].to_numpy(dtype=float),
        np.log(frequency_from_t1[freq_fit_mask].to_numpy(dtype=float)),
        1,
    )
    selected_frequency_t2 = np.exp(freq_intercept + freq_slope * ay_numeric.to_numpy(dtype=float))
else:
    selected_frequency_t2 = np.repeat(float(frequency_from_t1.mean()), len(ay_numeric))

if int(sev_fit_mask.sum()) >= 2:
    sev_slope, sev_intercept = np.polyfit(
        ay_numeric[sev_fit_mask].to_numpy(dtype=float),
        np.log(severity_from_t1[sev_fit_mask].to_numpy(dtype=float)),
        1,
    )
    selected_severity_t2 = np.exp(sev_intercept + sev_slope * ay_numeric.to_numpy(dtype=float))
else:
    selected_severity_t2 = np.repeat(float(severity_from_t1.mean()), len(ay_numeric))

selected_frequency_t2 = pd.Series(selected_frequency_t2, index=ultimate_counts_t1.index)
selected_severity_t2 = pd.Series(selected_severity_t2, index=ultimate_counts_t1.index)

ultimate_claims_t2 = exposure_proxy * selected_frequency_t2 * selected_severity_t2
ibnr_t2 = ultimate_claims_t2 - latest_claims

tech2_ay = pd.DataFrame(
    {
        'ExposureProxy': exposure_proxy.values,
        'SelectedFrequency_T2': selected_frequency_t2.values,
        'SelectedSeverity_T2': selected_severity_t2.values,
        'UltimateClaims_T2': ultimate_claims_t2.values,
        'LatestReportedClaims': latest_claims.values,
        'IBNR_T2': ibnr_t2.values,
    },
    index=ultimate_claims_t2.index.year,
)
tech2_ay.index.name = 'AccidentYear'
tech2_ay.loc['Total'] = tech2_ay.sum()
tech2_ay


## Frequency-Severity Method Comparison and Red-Flag Notes

**Advantages of Freq-Sev over pure development (chain ladder):**
- More stable for immature AYs — counts develop more smoothly than dollar amounts
- Provides insight into separate frequency and severity drivers
- Allows explicit incorporation of inflation (Technique #2 trend)
- Reduces leverage of chain ladder on very immature accident years

**Disadvantages:**
- Higher data requirements (count triangles, exposure data)
- Frequency and severity trends are additional assumptions with their own uncertainty
- Sensitive to incorrect trend selection (especially for Technique #2)

---

**EXAM RED FLAG —** In Technique #2, frequency and severity trends are fit independently from mature AYs. If fewer than 4–5 accident years are mature enough to calibrate the trend (development age ≥ 84 months), the slope estimate carries high uncertainty. On the exam, if n_mature_origins is small, treat Technique #2 projections for immature AYs as low credibility.

**EXAM RED FLAG —** The frequency and severity trends in Technique #2 are fit as independent log-linear regressions. If severity trend is driven by inflation that also affects frequency (e.g., more complex claims are both more numerous AND more expensive), the interaction is understated — the combined ultimate may be understated as a result.

**EXAM RED FLAG —** The Disposal Rate method assumes selected disposal rates by age are consistent across origin years. If settlement practices changed materially mid-triangle (e.g., a faster settlement strategy was adopted), the selected DR by age will blend inconsistent origin years — this is the same homogeneity problem that motivates Berquist-Sherman adjustment for paid triangles.

In [ ]:
final_summary = pd.DataFrame(
    {
        'Method': [
            'Freq-Sev Technique #1',
            'Freq-Sev Technique #2',
            'Disposal Rate Method',
        ],
        'TotalUltimateClaims': [
            float(ultimate_claims_t1.sum()),
            float(ultimate_claims_t2.sum()),
            float(disposal_projection.loc['Total', 'ProjectedUltimatePaidClaims']),
        ],
        'TotalIBNR': [
            float(ibnr_t1.sum()),
            float(ibnr_t2.sum()),
            float(disposal_projection.loc['Total', 'IBNR_DisposalMethod']),
        ],
    }
)

impact_table = pd.DataFrame(
    [
        [
            'Speedup in settlement rate',
            'Closed counts develop faster — count LDFs are lower than historical; ultimate count estimate drops; IBNR may be understated if not adjusted',
            'Disposal rates shift upward at early ages — selected DRs need to be from a consistent-settlement-rate period',
            'Frequency increases (more claims close faster) — Technique #2 captures this via trend if the change is gradual',
        ],
        [
            'Increase in claim severity (inflation)',
            'Severity triangle develops upward — severity LDFs increase; ultimate claims increase proportionally',
            'Incremental severity by development age increases — tail severity assumption increases; overall ultimate paid increases',
            'Severity trend slope captures the inflation if it falls within the calibration window; projected severity for immature AYs increases',
        ],
        [
            'Change in product mix (more complex claims)',
            'Severity increases; count closure rates may slow — both effects require re-evaluation of count and severity LDFs',
            'Disposal rates may shift lower (slower closure of complex claims)',
            'Frequency and severity both affected — may need to re-segment if mix change is material',
        ],
        [
            'Exposure growth',
            'Ultimate counts increase proportionally; ultimate claims increase; no explicit adjustment needed if severity is stable',
            'Not directly reflected in disposal rate projection (exposure is not explicit in the formula)',
            'Exposure enters directly in Technique #2 formula — ultimate claims grow proportionally with exposure',
        ],
        [
            'Average accident date shifts forward',
            'Count and severity CDFs understated at observed ages — less development observed in triangle window than would occur at later ages',
            'Disposal rates understate ultimate at each observed age',
            'Mature AY trend calibration may be optimistic for immature AYs if accident date shift is material',
        ],
        [
            'Sparse late-age data (long-tail line)',
            'Severity LDFs at late ages have low credibility — smoothing or external benchmarks needed',
            'Incremental severity at late ages is volatile — tail severity assumption carries high weight; key judgmental assumption',
            'Technique #2 avoids explicit late-age data by using trend extrapolation — may be preferred for long-tail lines',
        ],
    ],
    columns=['Change in Environment', 'Impact on Freq-Sev Techniques', 'Impact on Disposal Rate Method', 'Notes for Technique #2'],
)

print("Method comparison summary:")
display(final_summary)

print("\nEnvironment impact table:")
impact_table

In [ ]:
disp_triangle = cl.Triangle(
    disp_df,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Disposal Rate', 'Closed Claim Counts', 'Paid Claims'],
    cumulative=True,
)

dr_triangle = disp_triangle['Disposal Rate']
disp_closed_triangle = disp_triangle['Closed Claim Counts']
disp_paid_triangle = disp_triangle['Paid Claims']

disp_count_dev = cl.Development(average='volume', n_periods=-1).fit_transform(disp_closed_triangle)
disp_count_model = cl.Chainladder().fit(disp_count_dev)
ultimate_closed_counts = disp_count_model.ultimate_.to_frame().iloc[:, 0]

dr_long = dr_triangle.to_frame(origin_as_datetime=False, keepdims=True).reset_index()
dr_matrix = dr_long.pivot(index='origin', columns='development', values='Disposal Rate').sort_index().sort_index(axis=1)
disp_closed_long = disp_closed_triangle.to_frame(origin_as_datetime=False, keepdims=True).reset_index()
disp_closed_matrix = disp_closed_long.pivot(index='origin', columns='development', values='Closed Claim Counts').sort_index().sort_index(axis=1)
disp_paid_long = disp_paid_triangle.to_frame(origin_as_datetime=False, keepdims=True).reset_index()
disp_paid_matrix = disp_paid_long.pivot(index='origin', columns='development', values='Paid Claims').sort_index().sort_index(axis=1)

age_cols = disp_closed_matrix.columns.astype(int)
selected_dr_by_age = pd.Series(index=age_cols, dtype=float)
for age in age_cols:
    vals = dr_matrix[age].dropna()
    selected_dr_by_age.loc[age] = vals.tail(5).mean() if len(vals) >= 5 else vals.mean()
selected_dr_by_age = selected_dr_by_age.ffill().clip(upper=1.0)
if pd.isna(selected_dr_by_age.iloc[-1]) or selected_dr_by_age.iloc[-1] < 0.999:
    selected_dr_by_age.iloc[-1] = 1.0

inc_paid_matrix = disp_paid_matrix.diff(axis=1)
inc_paid_matrix.iloc[:, 0] = disp_paid_matrix.iloc[:, 0]
inc_closed_matrix = disp_closed_matrix.diff(axis=1)
inc_closed_matrix.iloc[:, 0] = disp_closed_matrix.iloc[:, 0]
inc_severity_matrix = inc_paid_matrix / inc_closed_matrix

selected_inc_severity_by_age = pd.Series(index=age_cols, dtype=float)
for age in age_cols:
    vals = inc_severity_matrix[age].replace([np.inf, -np.inf], np.nan).dropna()
    selected_inc_severity_by_age.loc[age] = vals.tail(5).median() if len(vals) >= 5 else vals.median()
selected_inc_severity_by_age = selected_inc_severity_by_age.ffill()

tail_mask = selected_inc_severity_by_age.index >= 108
if tail_mask.any() and selected_inc_severity_by_age[tail_mask].notna().any():
    selected_tail_severity = float(selected_inc_severity_by_age[tail_mask].dropna().mean())
    selected_inc_severity_by_age.loc[tail_mask] = selected_tail_severity

selected_dr_by_age, selected_inc_severity_by_age


## Disposal Projection Results and Tail-Severity Considerations

Tail-severity selection notes:
- Late maturities often have sparse counts and unstable incremental severities.
- A common practice is to blend late-age severities into a selected tail severity.
- The impact should be judged relative to how much claim count remains open at those ages.


In [ ]:
latest_closed = disp_closed_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_paid_disp = disp_paid_triangle.latest_diagonal.to_frame().iloc[:, 0]

latest_age_disp = disp_closed_matrix.notna().iloc[:, ::-1].idxmax(axis=1)
latest_age_disp.index = pd.to_datetime(latest_age_disp.index.astype(str) + "-01-01")
latest_age_disp = latest_age_disp.reindex(ultimate_closed_counts.index)

projection_rows = []
for idx in ultimate_closed_counts.index:
    ay = idx.year
    latest_age_ay = int(latest_age_disp.loc[idx])
    u_closed = float(ultimate_closed_counts.loc[idx])
    latest_paid_ay = float(latest_paid_disp.loc[idx]) if pd.notna(latest_paid_disp.loc[idx]) else 0.0

    if latest_age_ay in selected_dr_by_age.index:
        dr_prev = float(selected_dr_by_age.loc[latest_age_ay])
    else:
        dr_prev = float(selected_dr_by_age[selected_dr_by_age.index <= latest_age_ay].iloc[-1])

    projected_future_unpaid = 0.0
    for age in age_cols:
        age = int(age)
        if age <= latest_age_ay:
            continue
        dr_age = float(selected_dr_by_age.loc[age])
        projected_incremental_closed = max(u_closed * (dr_age - dr_prev), 0.0)
        selected_sev_age = float(selected_inc_severity_by_age.loc[age])
        projected_future_unpaid += projected_incremental_closed * selected_sev_age
        dr_prev = dr_age

    projected_ultimate_paid = latest_paid_ay + projected_future_unpaid
    projection_rows.append(
        {
            'AccidentYear': ay,
            'LatestAge': latest_age_ay,
            'LatestClosedCounts': float(latest_closed.loc[idx]),
            'ProjectedUltimateClosedCounts': u_closed,
            'LatestPaidClaims': latest_paid_ay,
            'ProjectedUltimatePaidClaims': projected_ultimate_paid,
            'IBNR_DisposalMethod': projected_ultimate_paid - latest_paid_ay,
        }
    )

disposal_projection = pd.DataFrame(projection_rows).set_index('AccidentYear').sort_index()
disposal_projection.loc['Total'] = disposal_projection.sum()
disposal_projection


In [ ]:
final_summary = pd.DataFrame(
    {
        'Method': [
            'Freq-Sev Technique #1',
            'Freq-Sev Technique #2',
            'Disposal Rate Method',
        ],
        'TotalUltimateClaims': [
            float(ultimate_claims_t1.sum()),
            float(ultimate_claims_t2.sum()),
            float(disposal_projection.loc['Total', 'ProjectedUltimatePaidClaims']),
        ],
        'TotalIBNR': [
            float(ibnr_t1.sum()),
            float(ibnr_t2.sum()),
            float(disposal_projection.loc['Total', 'IBNR_DisposalMethod']),
        ],
    }
)

final_notes = pd.DataFrame(
    [
        ['Technique #1', 'Separates counts and severity development directly; sensitive to severity selection.'],
        ['Technique #2', 'Adds explicit exposure/frequency/severity trend structure; stronger assumption load.'],
        ['Disposal Rate', 'Useful when closed-count emergence is credible and incremental severity is stable enough by maturity.'],
    ],
    columns=['Method', 'Interpretation'],
)

final_summary, final_notes


# Exam 5 Practice Problems -- Frequency-Severity Methods

Work through each problem by hand before running the solution cell.

This section covers three exam-testable formats:

| Format | Core Formula | Key Judgment |
|---|---|---|
| **1. Pure Freq-Sev** | Ult Claims x Ult Severity (both developed) | Separate development patterns |
| **2. Selected Severity** | Ult Claims x Selected Severity | Severity selection from mature data |
| **3. Exposure-Based** | Exposure x Frequency x Severity | Frequency per exposure unit |

> **Key principle:** Freq-Sev breaks loss into two components that can be diagnosed separately.
> Chain Ladder projects total losses directly and cannot isolate which component is driving change.

## Practice Problem 1: Pure Frequency-Severity (Fully Separate Projection)

You are given the following data as of December 31, 2024:

| AY | Reported Claims | Reported Losses | Claim Count CDF | Severity CDF |
|---|---|---|---|---|
| 2022 | 200 | 2,000,000 | 1.05 | 1.08 |
| 2023 | 160 | 1,440,000 | 1.15 | 1.12 |
| 2024 | 100 | 800,000 | 1.40 | 1.20 |

**(a)** Compute the current reported severity for each AY.

**(b)** Project ultimate claim counts using the Claim Count CDF.

**(c)** Project ultimate severity using the Severity CDF applied to **reported** severity.

**(d)** Calculate ultimate losses and IBNR for each AY.

**(e)** A large claim of $500,000 is included in AY 2023 reported losses.
What is the directional effect on this method's ultimate for AY 2023, and how does that compare to the same large claim's effect under Chain Ladder?

In [ ]:
import pandas as pd

# -- Data ----------------------------------------------------------------
data = {
    'AY':            [2022,      2023,      2024],
    'Rep_Claims':    [200,       160,       100],
    'Rep_Losses':    [2_000_000, 1_440_000, 800_000],
    'Count_CDF':     [1.05,      1.15,      1.40],
    'Severity_CDF':  [1.08,      1.12,      1.20],
}
df = pd.DataFrame(data)

# -- (a) Reported severity = Losses / Claims ------------------------------
df['Rep_Severity'] = df['Rep_Losses'] / df['Rep_Claims']

# -- (b) Ultimate claims --------------------------------------------------
df['Ult_Claims'] = df['Rep_Claims'] * df['Count_CDF']

# -- (c) Ultimate severity ------------------------------------------------
df['Ult_Severity'] = df['Rep_Severity'] * df['Severity_CDF']

# -- (d) Ultimate losses and IBNR -----------------------------------------
df['Ult_Losses'] = df['Ult_Claims'] * df['Ult_Severity']
df['IBNR'] = df['Ult_Losses'] - df['Rep_Losses']

print('=== Pure Frequency-Severity Projection ===')
cols = ['AY','Rep_Claims','Rep_Severity','Ult_Claims','Ult_Severity','Ult_Losses','IBNR']
print(df[cols].to_string(index=False, float_format='{:,.0f}'.format))

print()
print('Total IBNR:', f"{df['IBNR'].sum():,.0f}")

print()
print('--- Part (e): Large Loss Impact ---')
large_loss = 500_000
ult_sev_without = (1_440_000 - large_loss) / 160
ult_sev_with    = 1_440_000 / 160
print(f'Reported severity WITH large loss:    {ult_sev_with:,.0f}')
print(f'Reported severity WITHOUT large loss: {ult_sev_without:,.0f}')
print()
print('Freq-Sev: large loss inflates reported severity, which is then')
print('  multiplied by the Severity CDF to get ultimate severity.')
print('  The large loss is amplified by the Severity CDF (1.12 for AY 2023).')
print()
print('Chain Ladder: the large loss inflates the loss triangle at that')
print('  development age, and is similarly amplified by the loss CDF.')
print('  Effect is comparable in direction; Freq-Sev is not safer.')
print('  To mitigate: cap or remove large losses before developing, then add back.')


## Practice Problem 2: Projected Claim Count x Selected Ultimate Severity

Use the ultimate claim counts from Problem 1:

| AY | Ultimate Claims (from P1) | Reported Losses |
|---|---|---|
| 2022 | 210 | 2,000,000 |
| 2023 | 184 | 1,440,000 |
| 2024 | 140 | 800,000 |

The actuary selects an **ultimate severity of $10,500** based on mature AY data (AY 2022 has nearly converged; its developed severity is a reliable anchor).

**(a)** Calculate ultimate losses using the selected severity.

**(b)** Calculate IBNR. Compare to Problem 1 results.

**(c)** The selected severity ($10,500) is lower than the developed AY 2022 severity from Problem 1 ($10,800). Explain the trade-off in choosing a selected vs developed severity.

---

**Extension -- Exposure-Based:**
Earned exposures are 500 (2022), 520 (2023), 540 (2024).

**(d)** Compute the ultimate frequency (claims per exposure) for each AY.

**(e)** Select a frequency of **0.360 claims per exposure**. Using the selected severity of $10,500, compute exposure-based ultimate losses for each AY. Compare to part (a).

In [ ]:
import pandas as pd

# -- Data from P1 ---------------------------------------------------------
data = {
    'AY':           [2022,      2023,      2024],
    'Ult_Claims':   [210,       184,       140],
    'Rep_Losses':   [2_000_000, 1_440_000, 800_000],
    'Exposure':     [500,       520,       540],
    # Developed ultimates from P1 for comparison
    'P1_Ult':       [2_268_000, 1_854_720, 1_344_000],
}
df = pd.DataFrame(data)

selected_sev = 10_500
selected_freq = 0.360

# -- (a) & (b) Selected severity approach ---------------------------------
df['Ult_Selected'] = df['Ult_Claims'] * selected_sev
df['IBNR_Selected'] = df['Ult_Selected'] - df['Rep_Losses']
df['P1_IBNR'] = df['P1_Ult'] - df['Rep_Losses']

print('=== Method 2: Projected Counts x Selected Severity ===')
print(f'Selected ultimate severity: ${selected_sev:,.0f}')
print()
print(df[['AY','Ult_Claims','Ult_Selected','IBNR_Selected','P1_Ult','P1_IBNR']].to_string(
    index=False, float_format='{:,.0f}'.format))

print()
print('--- Part (c): Selected vs Developed Severity ---')
print('Selected severity ($10,500) < AY 2022 developed severity ($10,800).')
print('Advantage of selected: stable across all AYs, less sensitive to large losses')
print('  in any single AY. If AY 2022 had an unusual large claim, selected is safer.')
print('Disadvantage: ignores AY-specific development. If severity truly differs')
print('  by age/maturity, a single selected figure is too blunt.')
print('Also: selected severity must be trended if AYs span multiple policy years.')

print()
print('=== Extension: Exposure-Based Approach ===')
print(f'Selected frequency: {selected_freq} claims/exposure')
print(f'Selected severity:  ${selected_sev:,.0f}')
print()

# -- (d) Observed frequency from developed counts -------------------------
df['Obs_Freq'] = df['Ult_Claims'] / df['Exposure']

# -- (e) Exposure-based ultimate ------------------------------------------
df['Ult_Exposure'] = df['Exposure'] * selected_freq * selected_sev
df['IBNR_Exposure'] = df['Ult_Exposure'] - df['Rep_Losses']

print(df[['AY','Exposure','Obs_Freq','Ult_Claims','Ult_Exposure','IBNR_Exposure']].to_string(
    index=False, float_format='{:,.3f}'.format))

print()
print('Exposure-based uses a SELECTED frequency, not the AY-specific developed frequency.')
print('This is useful for growing books where older AYs have more mature data.')
print('AY 2024 exposure-based ultimate:', f"{df.loc[2,'Ult_Exposure']:,.0f}")
print('AY 2024 selected severity ultimate:', f"{df.loc[2,'Ult_Selected']:,.0f}")
print('Difference arises because exposure-based uses selected freq x exposure,')
print('  while selected severity uses developed claim counts from the triangle.')


## Practice Problem 3: Distortion Analysis and Method Comparison

**Scenario:** Claim reporting has accelerated. Claims that previously took 24 months to be reported are now reported within 12 months.

**(a)** What happens to the observed claim count development factors? (Higher, lower, or unchanged? Explain.)

**(b)** If you apply the old (pre-acceleration) development factors to the current reported counts, what happens to your projected ultimate claim counts?

**(c)** How does this affect the Freq-Sev ultimate losses? Is severity affected as well?

**(d)** Would Chain Ladder produce the same distortion? Explain whether the distortion shows up in the same direction.

---

**Scenario 2:** Your book has been growing rapidly -- earned exposures grew 25% in AY 2024.

**(e)** Which frequency-severity format (pure, selected severity, or exposure-based) best handles premium/exposure growth? Justify.

**(f)** A colleague proposes using Chain Ladder instead because 'it already handles growth through the triangle.' Evaluate this argument.

In [ ]:
print('=== Practice Problem 3: Model Answers ===')
print()

print('Part (a): Claim Reporting Acceleration')
print('  Faster reporting means MORE claims appear at early development ages')
print('  than historically. The LDF from 12->24 months will be SMALLER because')
print('  more claims are already in by 12 months. Observed age-to-age factors FALL.')
print()

print('Part (b): Applying old (higher) factors to current counts')
print('  Old factors assumed slow reporting -> fewer claims at early ages -> high LDF.')
print('  Current counts are HIGHER at early ages (acceleration). Multiplying by high')
print('  old factors OVERSTATES ultimate claim counts.')
print('  Result: projected ultimate claims too high -> IBNR overstated.')
print()

print('Part (c): Effect on Freq-Sev ultimate losses and severity')
print('  Frequency component: ultimate claims overstated (as above).')
print('  Severity component: if losses also emerged faster, reported severity may')
print('    appear higher at early ages. Severity LDFs may also be distorted.')
print('  Overall: Freq-Sev ultimate is overstated in BOTH components if factors')
print('    are not updated. Freq-Sev makes the distortion visible: you can see')
print('    that count factors have dropped, prompting factor revision.')
print()

print('Part (d): Does Chain Ladder show the same distortion?')
print('  Yes. CL projects total losses using historical LDFs. If reporting')
print('    accelerated, losses also emerge earlier -> CL LDFs fall.')
print('  Applying old (higher) CL LDFs to current reported losses similarly')
print('    overstates ultimate. Direction is the same: IBNR overstated.')
print('  Key difference: Freq-Sev allows you to DIAGNOSE whether the distortion')
print('    comes from counts, severity, or both. CL only tells you total losses changed.')
print()

print('Part (e): Best format for a growing book')
print('  Exposure-Based (Exposure x Frequency x Severity) is best for growth.')
print('  Reason: it explicitly accounts for the larger exposure base in AY 2024.')
print('  Pure Freq-Sev and Selected Severity both use developed claim counts from')
print('    the triangle, which reflect actual emerged claims, not full expected claims')
print('    for the grown exposure base.')
print('  Exposure-based uses external exposures -> adjusts for growth by design.')
print()

print('Part (f): Chain Ladder argument evaluation')
print('  Partially valid, but limited. CL does develop what is in the triangle,')
print('    but it anchors to REPORTED losses for the growing AY. If AY 2024 has')
print('    25% more exposure but similar development to prior years, CL will project')
print('    an ultimate that is proportional to AY 2024 reported -- which already')
print('    reflects growth to the extent claims have emerged.')
print('  However, if AY 2024 is immature, very few claims have emerged and CL')
print('    understates (or overstates with volatile data). Exposure-based directly')
print('    incorporates the known exposure count and is more reliable for immature,')
print('    high-growth AYs.')


## Conceptual Reference: When Each Format Works and Fails

### Method Comparison

| Dimension | Chain Ladder | Pure Freq-Sev | Selected Severity | Exposure-Based |
|---|---|---|---|---|
| Projects | Total losses | Claims x Severity (both developed) | Claims x selected sev | Exposure x Freq x Sev |
| Diagnostic power | Low | High | Moderate | High |
| Sensitive to large losses | Yes | Yes (via severity) | Less (selected) | Less (selected) |
| Handles growing book | No | No | Partial | Yes |
| Requires stable LDFs | Yes | Yes (two sets) | Counts only | Counts only |
| Requires exposure data | No | No | No | Yes |

### When Frequency-Severity is Preferred Over Chain Ladder

- Claim count and severity are driven by different forces (e.g., reform affects counts; inflation affects severity)
- Large losses are distorting total loss development
- You want to separately trend severity for calendar-year inflation
- Mix shifts have changed the types of claims (frequency changes but severity may be stable)
- Growing book where claim counts are more stable than total losses

### Distortion Sources -- Quick Reference

| Distortion | Effect on Frequency | Effect on Severity | Net Effect on Ultimate |
|---|---|---|---|
| Large loss in AY | Unchanged | Inflated | Overstated |
| Faster claim reporting | Higher early counts, lower LDFs | May inflate early sev | Overstated if old factors used |
| Slower closure (backlog) | Lower closed counts | Distorted incurred sev | Uncertain direction |
| Calendar-year inflation | None | Inflated in recent years | Overstated if sev not trended |
| Mix shift (more severe claims) | Unchanged | Inflated | Overstated |

### Quick Check: Reporting Acceleration

If claims are reported **faster** than historical patterns:
1. Observed count LDFs **fall** (more claims already in at early ages)
2. Applying old (higher) factors **overstates** ultimate counts
3. Severity LDFs may also **fall** (losses emerge earlier too)
4. Both components push ultimate **up** if old factors are used unchanged
5. Fix: re-derive development factors using only the post-acceleration period

### Severity Selection Checklist

When selecting an ultimate severity:
- [ ] Use only mature AYs where severity has converged
- [ ] Trend selected severity to the accident year being estimated
- [ ] Check for large loss contamination in the selected base
- [ ] Confirm mix of claim types is similar to the target AY
- [ ] Document the selection rationale (examiners credit explicit justification)


## Common Pitfalls / Exam Trap Awareness

### Mechanical Traps

| Trap | How to Avoid |
|---|---|
| Computing severity AFTER development instead of from reported | Reported severity = Reported Losses / Reported Claims; apply severity CDF to that |
| Applying CDF to losses instead of severity | Freq-Sev applies separate CDFs to counts and to severity -- never to total losses |
| Using paid counts with incurred losses (or vice versa) | Keep basis consistent: paid counts with paid losses, reported with reported |
| Forgetting to trend selected severity | A selected severity from AY 2020 applied to AY 2024 needs 4 years of trend |
| Claiming ultimate claims = reported claims x CDF then stop | Must ALSO develop severity separately; missing severity development understates ultimate |
| Confusing frequency (claims/exposure) with claim counts | Frequency needs exposure denominator; claim counts do not |

### Judgment / Written Answer Traps

| Trap | What Examiners Want |
|---|---|
| Saying 'use Freq-Sev when CL is unstable' without specifics | Name what is unstable: large losses inflating severity, or mix shift changing frequency |
| Ignoring large loss impact on severity | State direction: large loss inflates reported severity -> severity CDF amplifies it |
| Not mentioning trend when selecting severity | Always note whether selected severity needs trending and over what period |
| Saying Freq-Sev is always more accurate than CL | It requires TWO stable development patterns; if either is unstable, Freq-Sev breaks |
| Treating exposure-based as identical to selected severity | Exposure-based incorporates growth; selected severity does not without manual adjustment |
| Forgetting the diagnostic advantage | Freq-Sev lets you identify WHICH component (frequency or severity) is driving reserve change |

### Self-Check Before Finalizing

- [ ] Did I compute reported severity correctly (Losses / Claims, not Claims / Losses)?
- [ ] Are frequency and severity on the same basis (paid vs incurred)?
- [ ] Did I apply the Count CDF to counts AND the Severity CDF to severity (not one CDF to losses)?
- [ ] If I selected a severity, did I trend it to the target accident year?
- [ ] Did I check for large loss distortion in the severity data?
- [ ] For exposure-based: are exposures on a consistent basis across AYs?
- [ ] For written answers: did I name the specific distortion and its direction?
- [ ] Did I explain why Freq-Sev adds diagnostic value over Chain Ladder?
